In [1]:
# Contributors
# Alper Mumcular - am14533
# Anukriti Singh - as18692

In [2]:
# Importing libraries
import numpy as np
import math
import cv2
import matplotlib.pyplot as plt
import os

from PIL import Image
from typing import Union

In [3]:
def apply_sobel_operator(image: np.ndarray) -> np.ndarray:
    """
    Applies Sobel operator to compute the gradients in the x and y directions.

    Parameters:
    - image: np.ndarray - 2D input array (grayscale image)

    Returns:
    - G_x: np.ndarray - gradient in the x-direction
    - G_y: np.ndarray - gradient in the y-direction
    """
    # Sobel kernels
    sobel_x = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ], dtype=np.float32)

    sobel_y = np.array([
        [-1, -2, -1],
        [0, 0, 0],
        [1, 2, 1]
    ], dtype=np.float32)

    height, width = image.shape
    G_x = np.zeros_like(image, dtype=np.float32)
    G_y = np.zeros_like(image, dtype=np.float32)

    # Apply Sobel filter
    for i in range(1, height - 1):
        for j in range(1, width - 1):
            region = image[i-1:i+2, j-1:j+2]
            G_x[i, j] = np.sum(region * sobel_x)
            G_y[i, j] = np.sum(region * sobel_y)

    return G_x, G_y

In [4]:
def compute_harris_response(Gx: np.ndarray, Gy: np.ndarray, window_size: int, k: float) -> np.ndarray:
    """
    Computes the Harris corner response for each pixel using the gradient images.

    Parameters:
    - Gx: np.ndarray - Gradient of the image in the x-direction
    - Gy: np.ndarray - Gradient of the image in the y-direction
    - window_size: int - Size of the window used for computing the structure tensor
    - k: float - Harris detector free parameter, typically between 0.04 and 0.15

    Returns:
    - R_values: np.ndarray - Harris response matrix (same size as input image)
    """

    half = window_size // 2  # Half the window size for neighborhood
    height, width = Gx.shape
    R_values = np.zeros((height, width))  # Initialize response matrix

    for y in range(height):
        for x in range(width):
            A = np.zeros((2, 2))  # Structure tensor (2x2 matrix)

            # Loop through the local window around (x, y)
            for i in range(-half, half + 1):
                for j in range(-half, half + 1):
                    yy = y + i
                    xx = x + j

                    # Handle image boundaries
                    if 0 <= yy < height and 0 <= xx < width:
                        gx = Gx[yy, xx]
                        gy = Gy[yy, xx]
                    else:
                        gx = 0
                        gy = 0

                    # Accumulate structure tensor components
                    A += np.array([
                        [gx * gx, gx * gy],
                        [gx * gy, gy * gy]
                    ])

            A *= 2  # Since A(x, y) = 2 * A_W(x, y)

            # Compute Harris response: R = det(A) - k * (trace(A))^2
            det = np.linalg.det(A)
            trace = np.trace(A)
            R = det - k * (trace ** 2)

            R_values[y, x] = R  # Store response

    return R_values


In [5]:
def non_max_suppression(response: np.ndarray) -> np.ndarray:
    """
    Applies non-maximum suppression to a response matrix to retain only local maxima.

    Parameters:
    - response: np.ndarray - 2D array representing corner response values

    Returns:
    - suppressed: np.ndarray - 2D array with only local maxima retained; other values set to zero
    """

    height, width = response.shape
    suppressed = np.zeros_like(response)  # Initialize output array with zeros

    # Loop through each pixel excluding the border
    for y in range(1, height - 1):
        for x in range(1, width - 1):
            # Extract 3x3 neighborhood centered at (x, y)
            neighborhood = response[y-1:y+2, x-1:x+2]

            # Get the maximum value in the neighborhood
            max_value = np.max(neighborhood)

            # Retain the value only if it is the local maximum
            if response[y, x] == max_value:
                suppressed[y, x] = response[y, x]
            # All other values remain zero (suppressed)

    return suppressed

In [6]:
def calculate_harris_corner(image_path: str, image: np.ndarray, W: int, T: float, K: float,
                             is_gaussian: bool = False, gaussian_window: int = 5, sigma: float = 1) -> list:
    """
    Computes Harris corners from a grayscale image and saves the visualization as a BMP file.

    Parameters:
    - image_path: str - Path to the original image (used for saving results)
    - image: np.ndarray - Grayscale image array
    - W: int - Window size for the Harris response computation
    - T: float - Threshold factor (relative to max response) for corner selection
    - K: float - Harris detector parameter (typically 0.04 - 0.06)
    - is_gaussian: bool - Whether to apply Gaussian blur before processing
    - gaussian_window: int - Size of the Gaussian filter kernel
    - sigma: float - Standard deviation for the Gaussian filter

    Returns:
    - List of (row, column) tuples indicating detected corner coordinates
    """

    # Optionally apply Gaussian blur to the input image
    if is_gaussian:
        image = cv2.GaussianBlur(image, (gaussian_window, gaussian_window), sigma)

    # Compute image gradients
    Gx, Gy = apply_sobel_operator(image)

    # Compute Harris corner response matrix
    response = compute_harris_response(Gx, Gy, window_size=W, k=K)

    # Threshold the response to suppress weak corners
    threshold = T * response.max()
    response[response < threshold] = 0

    # Apply non-maximum suppression to refine corners
    nms = non_max_suppression(response)

    # Get coordinates of pixels where response exceeds threshold
    corners_i, corners_j = np.where(nms > threshold)

    # Prepare output file paths
    base_filename = os.path.basename(image_path)
    name, _ = os.path.splitext(base_filename)
    output_dir = "outputs"
    os.makedirs(output_dir, exist_ok=True)

    temp_png_path = os.path.join(output_dir, f"{name}_corners_temp.png")
    final_bmp_path = os.path.join(output_dir, f"{name}_corners.bmp")

    # Plot original image with detected corners overlayed
    fig, ax = plt.subplots(figsize=(image.shape[1] / 100, image.shape[0] / 100), dpi=100)
    ax.imshow(Image.open(image_path), cmap='gray')
    ax.plot(corners_j, corners_i, linestyle='None', marker='+', color='red', markersize=8, markeredgewidth=0.75)
    ax.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    # Save temporary PNG and convert to BMP format
    fig.savefig(temp_png_path, dpi=100, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    img = Image.open(temp_png_path)
    img.save(final_bmp_path)

    # Remove the temporary PNG file
    os.remove(temp_png_path)

    print(f"{final_bmp_path} is successfully created!")

    return list(zip(corners_i, corners_j))


In [7]:
def write_ascii_corners(filename: str, corners: list) -> None:
    """
    Writes the list of detected corners to a text file in ASCII format.

    Parameters:
    - filename: str - Path to the output text file
    - corners: list - List of (row, column) tuples representing corner coordinates

    Output Format:
    The first line contains the number of corners.
    Each subsequent line contains the i and j indices of a corner.
    """
    with open(filename, 'w') as f:
        # Write the number of corners
        f.write(f"{len(corners)}\n")

        # Write each corner coordinate
        for i, j in corners:
            f.write(f"{i} {j}\n")

    print(f"{filename} is successfully created!")

In [8]:
def region_correlation(left_image: np.ndarray, right_image: np.ndarray, left_points: list, right_points: list, W: int = 7):
    """
    Finds matching points between two stereo images using lecture slides

    Parameters:
    - left_image: np.ndarray - Grayscale left image
    - right_image: np.ndarray - Grayscale right image
    - left_points: list - List of (row, col) tuples representing feature points in the left image
    - right_points: list - List of (row, col) tuples representing feature points in the right image
    - W: int - Window size for the patch used in correlation (default is 7)

    Returns:
    - matches: list - List of tuples (left_point, best_right_point, correlation_score) for valid matches
    """

    shape = left_image.shape
    half = W // 2
    matches = []

    for left_point in left_points:
        # Skip if patch goes out of bounds in left image
        if left_point[0] - half < 0 or left_point[0] + half >= shape[0] or left_point[1] - half < 0 or left_point[1] + half >= shape[1]:
            continue

        # Extract and normalize the patch from left image
        f1 = left_image[left_point[0] - half : left_point[0] + half + 1,
                        left_point[1] - half : left_point[1] + half + 1]
        f1_hat = f1.mean()
        f1z = f1 - f1_hat
        f1_denominator = np.sqrt((f1z ** 2).sum())

        best_right_point = None
        r_best = 0  # Best correlation value

        for right_point in right_points:
            # Skip if row difference too large or patch goes out of bounds in right image
            if abs(right_point[0] - left_point[0]) >= 3 or \
               right_point[0] - half < 0 or right_point[0] + half >= shape[0] or \
               right_point[1] - half < 0 or right_point[1] + half >= shape[1]:
                continue

            # Skip if disparity is zero or if xl is less than xr
            if (right_point[1] >= left_point[1]):
                continue

            # Extract and normalize the patch from right image
            f2 = right_image[right_point[0] - half : right_point[0] + half + 1,
                             right_point[1] - half : right_point[1] + half + 1]
            f2_hat = f2.mean()
            f2z = f2 - f2_hat
            f2_denominator = np.sqrt((f2z ** 2).sum())

            # Compute normalized cross-correlation
            numerator = (f1z * f2z).sum()
            denominator = f1_denominator * f2_denominator
            r = numerator / denominator

            # Update best match if correlation improves
            if r > r_best:
                best_right_point = (right_point[0], right_point[1])
                r_best = r

        # Accept the match if correlation is strong enough
        if best_right_point is not None and r_best >= 0.8:
            matches.append((left_point, best_right_point, r_best))

    return matches


In [9]:
def compute_relative_depth(xl: float, xr: float, k: float = 1.0) -> float:
    """
    Computes the relative depth z' given x-coordinates in left and right stereo images.

    Parameters:
    - xl: float - x-coordinate of a feature in the left image
    - xr: float - x-coordinate of the corresponding feature in the right image
    - k: float - camera constant or baseline*focal_length (default is 1.0)

    Returns:
    - z_prime: float - relative depth, inversely proportional to disparity

    Raises:
    - ValueError: if disparity (xl - xr) is zero, which would cause division by zero
    """
    disparity = xl - xr
    if disparity == 0:
        raise ValueError("Disparity is zero: Cannot compute depth due to division by zero.")
    z_prime = k / disparity
    return z_prime


In [10]:
def write_ascii_matches(filename: str, matched: list, z_prime_prime: list) -> None:
    """
    Writes matched feature points along with their correlation scores and relative depths to a text file.

    Parameters:
    - filename: str - Path to the output text file
    - matched: list - List of tuples in the form ((y1, x1), (y2, x2), correlation)
                      representing matched points between left and right images
    - z_prime_prime: list - List of relative depth values corresponding to each match

    Output Format:
    The first line contains the number of matches.
    Each subsequent line contains:
    y1 x1 y2 x2 correlation z''
    (coordinates of the point in the left and right images, correlation score, and computed depth)
    """

    i = 0
    with open(filename, 'w') as f:
        # Write number of matches
        f.write(f"{len(matched)}\n")

        # Write each match with its corresponding depth
        for m in matched:
            f.write(f"{m[0][0]} {m[0][1]} {m[1][0]} {m[1][1]} {m[2]:.4f} {z_prime_prime[i]}\n")
            i += 1

    print(f"{filename} is successfully created!")

In [11]:
def rescale_relative_depth(z_prime_values: np.ndarray) -> np.ndarray:
    """
    Rescales a numpy array of relative depth values z' to the range [10, 255].

    Parameters:
    - z_prime_values: np.ndarray - array of relative depth values

    Returns:
    - np.ndarray - rescaled and rounded depth values z''

    Raises:
    - ValueError: if max and min equal to each other, it would cause division by zero
    """
    z_min = np.min(z_prime_values)
    z_max = np.max(z_prime_values)

    if z_max == z_min:
        raise ValueError("Error: Division by zero.")

    # Applying the given formula in homework file
    z_scaled = 255 - np.floor(((245 * (z_prime_values - z_min)) / (z_max - z_min)) + 0.5).astype(int)
    return z_scaled.astype(int)


In [12]:
def build_and_save_relative_depth_map(matches: list, right_image_shape: tuple, output_match_path: str, output_depth_path: str) -> None:
    """
    Builds a sparse relative-depth map from stereo matches and saves both
    the match information and the resulting depth map.

    Parameters:
    - matches: List[Tuple[Tuple[int, int], Tuple[int, int], float]] -
        List of matched point pairs and their correlation scores in the form:
        [((i1, j1), (i2, j2), score), ...]
    - right_image_shape: Tuple[int, int] - Shape of the right image (height, width)
    - output_match_path: str - Path to save ASCII match data
    - output_depth_path: str - Path to save the depth BMP image
    """

    # Initialize depth map with zeros
    depth = np.zeros(right_image_shape)

    # Compute relative depths (z') from x-coordinates of the matched points
    z_primes = np.array([
        compute_relative_depth(j1, j2) for ((_, j1), (_, j2), _) in matches
    ])

    # Normalize depth values for visualization (e.g., scale to 0–255)
    z_primes_rescaled = rescale_relative_depth(z_primes)

    # Write match data with correlation and depth to ASCII file
    write_ascii_matches(output_match_path, matches, z_primes_rescaled)

    # Assign depth values to corresponding pixel locations
    for idx, ((i1, j1), _, _) in enumerate(matches):
        depth[i1, j1] = z_primes_rescaled[idx]

    # Convert depth map to uint8 and save as BMP
    depth_uint8 = depth.astype(np.uint8)
    Image.fromarray(depth_uint8).save(output_depth_path)

    print(f"{output_depth_path} is successfully created!")


Parameter Tested Values:

W = 5,7

T = 0.005, 0.01, 0.015, 0.02, 0.025

K = 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15

In [13]:
# Image paths
image_path = ["ball_left_new.bmp", "ball_right_new.bmp", "Moebius_left.bmp", "Moebius_right.bmp", "Outdoor_left.bmp", "Outdoor_right.bmp"]

# Load images
left_image = Image.open(image_path[0])
left_image = np.array(left_image)

right_image = Image.open(image_path[1])
right_image = np.array(right_image)

# Calculate Harris corners
left_corners = calculate_harris_corner(image_path[0], left_image, W=7, T=0.01, K=0.12)
right_corners = calculate_harris_corner(image_path[1], right_image, W=7, T=0.01, K=0.12)

# Write corner points to ASCII files
write_ascii_corners("outputs/ball_left_features.txt", left_corners)
write_ascii_corners("outputs/ball_right_features.txt", right_corners)

# Perform region correlation to find matches
matches = region_correlation(left_image, right_image, left_corners, right_corners, 7)

# Build and save the relative depth map and match data
build_and_save_relative_depth_map(matches=matches, right_image_shape=right_image.shape, output_match_path="outputs/ball_matches.txt", output_depth_path="outputs/ball_depth.bmp")


outputs/ball_left_new_corners.bmp is successfully created!
outputs/ball_right_new_corners.bmp is successfully created!
outputs/ball_left_features.txt is successfully created!
outputs/ball_right_features.txt is successfully created!
outputs/ball_matches.txt is successfully created!
outputs/ball_depth.bmp is successfully created!


In [14]:
# Load images
left_image = Image.open(image_path[2])
left_image = np.array(left_image)

right_image = Image.open(image_path[3])
right_image = np.array(right_image)

# Calculate Harris corners
left_corners = calculate_harris_corner(image_path[2], left_image, W=7, T=0.01, K=0.08, is_gaussian=True, gaussian_window=7, sigma=1)
right_corners = calculate_harris_corner(image_path[3], right_image, W=7, T=0.01, K=0.08, is_gaussian=True, gaussian_window=7, sigma=1)

# Write corner points to ASCII files
write_ascii_corners("outputs/Moebius_left_features.txt", left_corners)
write_ascii_corners("outputs/Moebius_right_features.txt", right_corners)

# Perform region correlation to find matches
matches = region_correlation(left_image, right_image, left_corners, right_corners, 7)

# Build and save the relative depth map and match data
build_and_save_relative_depth_map(matches=matches, right_image_shape=right_image.shape, output_match_path="outputs/Moebius_matches.txt", output_depth_path="outputs/Moebius_depth.bmp")

outputs/Moebius_left_corners.bmp is successfully created!
outputs/Moebius_right_corners.bmp is successfully created!
outputs/Moebius_left_features.txt is successfully created!
outputs/Moebius_right_features.txt is successfully created!
outputs/Moebius_matches.txt is successfully created!
outputs/Moebius_depth.bmp is successfully created!


In [15]:
# Load images
left_image = Image.open(image_path[4])
left_image = np.array(left_image)

right_image = Image.open(image_path[5])
right_image = np.array(right_image)

# Calculate Harris corners
left_corners = calculate_harris_corner(image_path[4], left_image, W=7, T=0.015, K=0.08, is_gaussian=True, gaussian_window=5, sigma=1)
right_corners = calculate_harris_corner(image_path[5], right_image, W=7, T=0.015, K=0.08, is_gaussian=True, gaussian_window=5, sigma=1)

# Write corner points to ASCII files
write_ascii_corners("outputs/Outdoor_left_features.txt", left_corners)
write_ascii_corners("outputs/Outdoor_right_features.txt", right_corners)

# Perform region correlation to find matches
matches = region_correlation(left_image, right_image, left_corners, right_corners, 7)

# Build and save the relative depth map and match data
build_and_save_relative_depth_map(matches=matches, right_image_shape=right_image.shape, output_match_path="outputs/Outdoor_matches.txt", output_depth_path="outputs/Outdoor_depth.bmp")

outputs/Outdoor_left_corners.bmp is successfully created!
outputs/Outdoor_right_corners.bmp is successfully created!
outputs/Outdoor_left_features.txt is successfully created!
outputs/Outdoor_right_features.txt is successfully created!
outputs/Outdoor_matches.txt is successfully created!
outputs/Outdoor_depth.bmp is successfully created!
